# Python Interface for SenseAI's BPFA Algorithm

**Version 1.2** </br>
02-07-2026

*For questions: luco.buise@radboudumc.nl*

In [ ]:
import os

import pickle
import tifffile
import cv2
import h5py

import numpy as np
import arrayfire as af
from senseai import SenseAI
from senseai.bpfa import BPFAReconstruction

from PIL import Image

from bpfa_utils import perform_reconstruction

from bpfa_ui import BPFAControls

## File Selection

Select a ``.tiff`` file. 

In [ ]:
## CHANGE FILE HERE
filepath_sparse = R"C:\Users\Image Analysis\Documents\SenseAI\Workspaces\24808\24808_20260605\cryoU-grid-60min-pasp-_33_HAADF_18_1.212_2_1.7_33.tiff"

# Configure SenseAI Instance
senseai = SenseAI("C:\Program Files\SenseAI\SenseAI 2026.4.1\SenseAI.dll")
af.set_backend("cuda")

# transform to ArrayFire arrays
input_array = af.interop.from_ndarray(tifffile.imread(filepath_sparse).astype(np.float32)[2,:,:]).as_type(af.Dtype.f32)/65535.0 ## changed for .tiff files
mask_array = af.interop.from_ndarray(tifffile.imread(filepath_sparse).astype(np.float32)[3,:,:]).as_type(af.Dtype.f32)/65535.0


## Configure BPFA Parameters

Change the parameters below as you like, these will be the intitial values of your algorithm. The BPFA object algorithm will be instantiated immediately afterwards. You will be able to change the following parameters live while running the algorithm:
* *K*
* *Limit*
* *LR*
* *Ones Element*
* *Avg Element*
* *Batchsize*

In [ ]:
parameters = {
    "patch_shape" : (18,18,1,1), #dict patch shapes
	"K" : 42,
	"limit" : 13,
	"dict_type" : 'Random',
	"order" : 'Random',
	"g_e" : 10.0,
	"g_w" : 0.26,
	"g_d" : 64.0,
	"lr" : 0.95,
	"pi_a" : 0.0,
	"pi_b" : 1.0,
	"ones_element" : True,
	"avg_element" : False
}

batch_size = 42000

bpfa = BPFAReconstruction(input_array, **parameters) # initialisation of BPFA instance

## Run the Algorithm

Running the cell below will start the BPFA algorithm. The sliders allow you to change parameters as your see fit while the dictionary and sparse coding are training. The reconstruction is updated live, allowing you to see the changes to both the dictionary and the reconstruction as you make changes. You can also plot the metrics quantifying the reconstruction, as well as visualise the weights belonging to the dictionary elements. Keep in mind that calculating these could increase training time.

You can click on an atom in the dictionary window to remove it. Note that changing the number of dictionary elements (either by removing them or changing *K*) will reset the sparse coding weights.

It might sometimes be the case that a change you make throws an exception. The error-handling should (hopefully) work well enough to handle almost all exceptions and allow you to continue without issues.

**ALWAYS** stop the algorithm using the ``Stop`` button, not by interrupting the kernel. Otherwise your kernel could crash and you will lose the trained data. If you want to continue training afterwards, you can just run the cell again.

In [ ]:
controls = BPFAControls(bpfa, batch_size) # initialise the UI

perform_reconstruction(bpfa, input_array, mask_array, controls) # start the algorithm

# Close All Windows

Run the cell below if you want to close the plot, dictionary and reconstruction windows.

In [ ]:
cv2.destroyAllWindows()

# Saving Dictionary & Reconstruction

Use the cells below to save your data manually, rather than through the ``save all`` button. The dictionary will be saved as a ``.pkl`` file, and the reconstruction as a ``.tiff`` file. You can find them in the directories `pickled_dicts` and `recons` respectively.

Change the ``f_name`` as you like. 

In [ ]:
## SAVING BPFA-DICTS AS PICKLE
curr_bpfa_dict = bpfa.D.to_ndarray() # transform to np array first
pickle_dir = "pickled_dicts"
f_name = "20-n1" # <------------------ CHANGE NAME HERE
path = os.path.join(pickle_dir, f_name  + ".pkl")

with open(path, "wb") as f:
    pickle.dump(curr_bpfa_dict, f)

In [ ]:
## SAVE RECONSTRUCTION AS TIFF
recon_dir = "recons"
f_name_recon = "recon_chip_0h_05" # <------------------ CHANGE NAME HERE
path = os.path.join(recon_dir, f_name_recon + ".tiff")

reconstruction = bpfa.reconstruct(True, False, "FullW")
reconstruction_arr = reconstruction.to_ndarray()
im = Image.fromarray(reconstruction_arr)
im.save(path)

## Opening a Saved Dictionary

If you want to work with a previously saved dictionary, you can open the `.pkl` or `.hdf5` file using the code below. You can then pass the new dictionary to the BPFA instance you initialised earlier.

In [ ]:
## OPEN BPFA-DICTS FROM PICKLE
pickle_dir = "pickled_dicts"
f_name = "20-n1" # <------------------ CHANGE NAME HERE
path = os.path.join(pickle_dir, f_name + ".pkl")

with open(path, "rb") as f:
    bpfa_dict = pickle.load(f)

In [ ]:
## OPEN BPFA-DICT FROM HDF5
hdf5_dir = "old_dicts"
f_name = "20251111_img27olddict" # <------------------ CHANGE NAME HERE
path = os.path.join(hdf5_dir, f_name + ".hdf5")

dict_file_hdf5 = h5py.File(path, 'r')
bpfa_dict = dict_file_hdf5["Dictionaries"]['Dictionary 1']['Data'][:].T # transpose to match pickled shape of patch_shape**2 x K

In [ ]:
## GIVE OPENED DICT TO BPFA OBJECT
bpfa.K = bpfa_dict.shape[1] # adjust K accordingly
bpfa.D = af.interop.from_ndarray(bpfa_dict[:])

## BPFA Sparse Only

If you want to use a new dictionary, you might not want it to change, and only learn the sparse coding weights. The code below runs the BPFA algorithm as above, but it will skip the dictionary learning step, only doing sparse coding and reconstructing. 

If you're fine with your new dictionary changing, you can run the algorithm in the cell above just as before.

In [ ]:
controls = BPFAControls(bpfa, batch_size) # initialise the UI

perform_reconstruction(bpfa, input_array, mask_array, controls, only_sparse=True) # start the algorithm (sparse-only)